### Reading the data

In [ ]:
import numpy as np, pandas as pd
#####################################################
#data file
file_name='raw_data.csv'
#data file spearator
sep=','
#set the column in the data file where the smiles can be found
smiles_column='smiles'
#properties
properties=['normal','poisson','uniform','gamma1','gamma2','gamma3']
#####################################################
df= pd.read_csv(file_name,sep=sep, na_values = ['NAN', '?','NaN'])
df.head()

### Removing Qualifiers

In this cell we provide some options to handle common qualifiers. 

* **~**: ~ is plainly dropped from the value. ~0.2 -> 0.2 
* **+**: + is plainly dropped from the value. +0.2 -> 0.2
* **<**: Here we provide two options, simply drop: <0.2 -> 0.2 or you can provide a distribution p and then we subtract a value form this distribution: <0.2 -> 0.2-p_i
* **>**: Here we provide two options, simply drop: >0.2 -> 0.2 or you can provide a distribution p and then we add a value form this distribution: >0.2 -> 0.2+p_i

For a value containing a list separated by *;*, we provide options to take the mean or the median.

We provide a few example distributions. Note that in the case of the normal distribution, the value drawn might be negative.

In [ ]:
from numpy.random import default_rng
from functools import partial

class normal_wrapper:
    def __init__(self,mu : float =0,sigma : float =1,rng=None):
        self.mu=mu
        self.sigma=sigma
        if rng is None:
            self.rng=default_rng()
        else:
            self.rng=rng
    
    def __call__(self) -> float:
        return self.rng.normal(self.mu, self.sigma, 1).item() 
    
class uniform_wrapper:
    def __init__(self,low : float =0,high : float =1,rng=None):
        self.low=low
        self.high=high
        if rng is None:
            self.rng=default_rng()
        else:
            self.rng=rng
    
    def __call__(self) -> float:
        return self.rng.uniform(self.low, self.high, 1).item() 
    
class gamma_wrapper:
    def __init__(self,shape : float=1.0,scale : float =1,rng=None):
        assert shape > 0 and scale > 0 , 'shape and scale must both be >0'
        self.shape=shape
        self.scale=scale
        if rng is None:
            self.rng=default_rng()
        else:
            self.rng=rng
    
    def __call__(self) -> float:
        return self.rng.gamma(self.shape, self.scale, 1).item() 
    
def transform_qualifier(n : str, list_option : str ='mean', noise_gen=None) -> float:
    assert list_option in ['mean','median'],'provide valid list_option ([mean,median])'
    n= n.replace('~','').replace('+','')
    
    #use provided noise generator to add or subtract for respectively > or < qualifiers
    if noise_gen is not None:
        if ';' in n:
            n = [float(x[1:])+noise_gen() if x.startswith('>') else float(x[1:])-noise_gen() if x.startswith('<') else float(x.replace('~','').replace('+','')) for x in n.split(';')]
            if list_option=='mean':
                return sum(n) / len(n)
            elif list_option=='median':
                return float(np.median(n))
        elif n.startswith('>'):
            return float(n[1:])+noise_gen() 
        elif n.startswith('<'):
            return float(n[1:])-noise_gen() 
            
    n = n.replace('>', '').replace('<', '')
    if n:
        if ';' in n:
            n = [float(x) for x in n.split(';')]
            if list_option=='mean':
                return sum(n) / len(n)
            elif list_option=='mean':
                return float(np.median(n))
        return float(n)
    return float('nan')

df['normal_normal']=df['normal'].apply(partial(transform_qualifier, list_option='mean', noise_gen=normal_wrapper()))
df['normal_uniform']=df['normal'].apply(partial(transform_qualifier, list_option='mean', noise_gen=uniform_wrapper()))
df['normal_gamma']=df['normal'].apply(partial(transform_qualifier, list_option='mean', noise_gen=gamma_wrapper()))
df['poisson']=df['poisson'].apply(partial(transform_qualifier))
df['uniform_mean']=df['uniform'].apply(partial(transform_qualifier, list_option='mean', noise_gen=uniform_wrapper(0,0.1)))
df['uniform_median']=df['uniform'].apply(partial(transform_qualifier, list_option='median', noise_gen=uniform_wrapper(0,0.1)))
df.head()

### Plot histogram


In [ ]:
import plotly.graph_objects as go
#function to create plotly histogram
def create_plotly_hist(df,property_to_be_plotted,fig_size=(800,800)):
    mask=np.isnan(df[property_to_be_plotted])
    chart_data,bin_edges=np.histogram(df[property_to_be_plotted][~mask], bins=nb_bins,density=True)
    label_step=1
    if nb_bins>20:
        label_step=2
    if nb_bins>100:
        label_step=5
    fig = go.Figure(data=[
    go.Bar(name=property_to_be_plotted,
           x=np.round(bin_edges[1:],2),
           y=chart_data,
           text=np.round(chart_data,3),
           hovertemplate='<b>Prob. dens.</b>: %{y:.3f}',
           textposition='inside',
           textfont_size=10,
           textfont_color="black"
          )])
    xlabels=np.round(bin_edges[1::label_step],2)
    fig.update_layout(title=f'Histogram with {nb_bins} bins<br><sup>Property: {property_to_be_plotted}</sup>',
                title_font_size=18,height=fig_size[1], width=fig_size[0],
               xaxis_title=f'{property_to_be_plotted} bins',
               yaxis_title=f'probability density function values',
               xaxis_tickangle=-45,
               xaxis = dict(
                    tickmode = 'array',
                    tickvals = xlabels,
                    ticktext = xlabels
                ),
                legend=dict(
                    title_text='',
                    traceorder="normal",
                    font=dict(
                        size=11,
                        color="black"
                        )
                    )
             )
    return fig,xlabels

Here we show a histogram for a specified property

In [ ]:
#set number of bins
nb_bins=20
#set property
property_to_be_plotted='gamma2'
fig,xlabels=create_plotly_hist(df,property_to_be_plotted,fig_size=(1000,800))
fig.show()    

### Fitting distributions
In this section we are using the library fitter to fit some common distributions on the specified property.

fitter uses the distributions from scipy.

In [ ]:
!pip install --user fitter

In [ ]:
from fitter import Fitter, get_common_distributions, get_distributions
def fit_distributions(df,property_to_be_plotted,distributions=None):
    mask=np.isnan(df[property_to_be_plotted])
    fitting_values = df[property_to_be_plotted][~mask].values
    if distributions is None:
        distributions=['cauchy', 'chi2', 'expon', 'exponpow', 'gamma', 'lognorm', 'norm', 'powerlaw', 'rayleigh', 'uniform']#,'burr','beta']
    f = Fitter(fitting_values,distributions=distributions)
    f.fit()
    return f
#fit distributions for property_to_be_plotted
f=fit_distributions(df,property_to_be_plotted)
f.summary(Nbest=3)

Add fitted pdf to histogram

In [ ]:
import scipy.stats

#adjust function based on output of fit_distributions
dist = scipy.stats.gamma
param = f.fitted_param['gamma']

pdf_fitted = dist.pdf(xlabels, *param)
fig.add_trace(go.Scatter(x=xlabels, y=pdf_fitted,
                    mode='lines+markers',
                    name='fitted pdf',
                    marker_symbol='circle',
                    marker=dict(color=f'rgba(255, 0, 0,0.8)',line_width=0.5, size=4)))
fig.show()

### Transforming Data

In this cell we transform the data using some common transformations. At the end, we use the fitting software again to compute the best fits for the boxcox transformation. 

In [ ]:
from scipy.stats import boxcox
from scipy.stats import yeojohnson

def p_scale(in_value: float, in_scale : str) -> float:
    '''Converts float to negative log10 scale'''
    if 'uM' in in_scale:
        return 6-np.log10(in_value)
    elif 'nM' in in_scale:
        return 9-np.log10(in_value)
    elif 'M' in in_scale:
        return -np.log10(in_value)

df[f'log10_{property_to_be_plotted}'] = np.log10(df[property_to_be_plotted])
df[f'sqrt_{property_to_be_plotted}']  = df[property_to_be_plotted]**(1/2)
df[f'1over_{property_to_be_plotted}']  = 1/df[property_to_be_plotted]
df[f'pscale_{property_to_be_plotted}'] =df[property_to_be_plotted].apply(partial(p_scale,in_scale='M'))
df[f'boxcox_{property_to_be_plotted}'], lam = boxcox(df[property_to_be_plotted])
df[f'yeojohnson_{property_to_be_plotted}'], lam = yeojohnson(df[property_to_be_plotted])

fig_transfo,xlabels=create_plotly_hist(df,f'boxcox_{property_to_be_plotted}',fig_size=(1000,800))
f=fit_distributions(df,f'boxcox_{property_to_be_plotted}')
f.summary(Nbest=3)

Add fitted pdf to histogram

In [ ]:
#select normal distribution
dist = scipy.stats.norm
param = f.fitted_param['norm']

pdf_fitted = dist.pdf(xlabels, *param)
fig_transfo.add_trace(go.Scatter(x=xlabels, y=pdf_fitted,
                    mode='lines+markers',
                    name='fitted pdf',
                    marker_symbol='circle',
                    marker=dict(color=f'rgba(255, 0, 0,0.8)',line_width=0.5, size=4)))


#adding interval markers
mu=np.mean(df[f'boxcox_{property_to_be_plotted}'])
sigma=np.std(df[f'boxcox_{property_to_be_plotted}'])
for confidence in [1,2,3,3.8]:
    fig_transfo.add_vline(x=mu+confidence*sigma,line=dict(color=f'rgba(112,128,144,0.8)', width=2, dash='dash'),
                 annotation_text=f'&mu;+{confidence}\u03C3={np.round(mu+confidence*sigma,2)}',
                 annotation_font_size=12,
                 annotation_position="top right")
    fig_transfo.add_vline(x=mu-confidence*sigma,line=dict(color=f'rgba(112,128,144,0.8)', width=2, dash='dash'),
                 annotation_text=f'&mu;-{confidence}\u03C3={np.round(mu-confidence*sigma,2)}',
                 annotation_font_size=12,
                 annotation_position="top right")
fig_transfo.add_vline(x=mu,line=dict(color=f'rgba(112,128,144,0.8)', width=2, dash='dash'),
                 annotation_text=f'&mu;={np.round(mu,2)}',
                 annotation_font_size=12,
                 annotation_position="top right")

fig_transfo.update_yaxes(range=[0, max(pdf_fitted)+0.1])
    
fig_transfo.show()

### Outlier removal based on normal distribution

In this section we provide a function to delete outliers based on the normal distribution. Basicly we set values to nan if they are outside the interval [&mu;-confidence\*&sigma;, &mu;+confidence*&sigma;] with confidence a chosen variable, &mu; the mean of the target and &sigma; the standard deviation of the target. 

In the example we use both the gamma2 and the boxcox transformed property. Note that the underlying distribution of gamma2 is not normal and the outliers removed are not actually outliers as illustrated by the boxcox transformed property (no outliers). This function should only be used for normal distributed targets. 

In [ ]:
from typing import List
#confidence
confidence=3.8

def normal_distr_outlier_removal(df : pd.DataFrame,properties : List[str],smiles_column : str,confidence : float) -> pd.DataFrame:   
    for c in properties:
    #removing samples outside interval
        df[c] = pd.to_numeric(df[c], errors='coerce')
        m=df[c].mean()
        sd= df[c].std()
        nb_outliers=len(df[abs(df[c] -m) > confidence*sd])
        print(f'outliers removed in {c}: {nb_outliers}')
        if nb_outliers>0:
            print(f'outliers {c}: \n'+',\n'.join([f'{smi}:{val}' for smi,val in zip(df.loc[abs(df[c] -m) > confidence*sd, smiles_column],df.loc[abs(df[c] -m) > confidence*sd, c])])) 
        #print(c, m, sd)
        df.loc[abs(df[c] -m) > confidence*sd, c]=np.nan
    return df

df=normal_distr_outlier_removal(df,['gamma2',f'boxcox_{property_to_be_plotted}'],smiles_column,confidence) 

### Outlier removal based on Bottleneck features of standardized Smiles

Using the encoder, we can generate the Bottleneck features for the smiles. First these are standardized and next the feature matrix is created in X.

Using this feature matrix X, we can fit outlier detection methods from scikit-learn, namely LocalOutlierFactor and IsolationForest.

Using the outliers predictions from these methods, we plots the chemical structures on 2D coordinates both for PCA with 2 components and using TSNE. For TSNE the number of features of data matrix X is first reduced to 50 before fitting TSNE. 

The outliers, detected here, are outliers in the provided data that the method thinks are outliers based on the Bottleneck generated features. In the end you have to select the smiles which are removed. This is only an indication, not a requirement. 

In [ ]:
from automol.feature_generators import retrieve_default_offline_generators
from automol.property_prep import add_stereo_smiles,validate_rdkit_smiles, add_rdkit_standardized_smiles
import pandas as pd, numpy as np
df= pd.read_csv('clustered_PPB_Cyprotex.csv',sep=',', na_values = ['NAN', '?','NaN'])
df['smiles']=df['original_smiles']

rdkit_standardization=True

if rdkit_standardization:
    add_rdkit_standardized_smiles(df, 'smiles',verbose=1,outname='standardized smiles')
else:
    add_stereo_smiles(df,'smiles',verbose=1,outname='standardized smiles')

df.dropna(inplace=True, subset = ['standardized smiles'])
#dictionary holding the default feature generators
feature_generators=retrieve_default_offline_generators( radius=2,nbits=2048)

smiles_list=df['standardized smiles'].values
X=feature_generators['Bottleneck'].generate(smiles_list)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest

clf = LocalOutlierFactor(n_neighbors=20, contamination=0.1)
# use fit_predict to compute the predicted labels of the training samples
# (when LOF is used for outlier detection, the estimator has no predict,
# decision_function and score_samples methods).
y_pred_LOF = clf.fit_predict(X)

rng = np.random.RandomState(42)
clf = IsolationForest(max_samples=100, random_state=rng)
clf.fit(X)
y_pred_IsoForest = clf.predict(X)
iso_l=[]
lof_l=[]
both_l=[]

for outl_lof,outl_iso,smi,index in zip(y_pred_LOF,y_pred_IsoForest,smiles_list,df.index):
    if outl_lof==-1:
        lof_l+=[index]
    if outl_iso==-1:
        iso_l+=[index]
    if outl_lof==-1 and outl_iso==-1:
        both_l+=[index]
        print(f'SMILES: {smi} is considered an outlier in the dataset by both algorithms')
iso_cnt=len(iso_l)
lof_cnt=len(lof_l)
both_cnt=len(both_l)
print(f'LocalOutlierFactor detected {lof_cnt} SMILES as outliers.\nIsolationForest detected {iso_cnt} SMILES as outliers.\nThe number of common outliers was {both_cnt}.') 

In [ ]:
colors=['red' if val in both_l else 'darkorange' if val in iso_l else 'mediumblue' if val in lof_l else 'darkgreen' for val in df.index]
labels=['IsoForest+LOF' if val in both_l else 'IsoForest' if val in iso_l else 'LOF' if val in lof_l else 'Valid Sample' for val in df.index]

In [ ]:
from bokeh.io import output_notebook
output_notebook()
from rdkit import Chem
from rdkit.Chem import MolFromSmiles, AllChem
from rdkit import  DataStructs
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Chem.PandasTools import ChangeMoleculeRendering
from IPython.display import SVG
from bokeh.plotting import figure, show, output_notebook, ColumnDataSource
from bokeh.models import HoverTool
from bokeh.transform import factor_cmap
from bokeh.plotting import figure, output_file, save
from bokeh.models import Title
from bokeh.models import Span
from bokeh.models import Legend, LegendItem

def _prepareMol(mol,kekulize):
    mc = Chem.Mol(mol.ToBinary())
    if kekulize:
        try:
            Chem.Kekulize(mc)
        except:
            mc = Chem.Mol(mol.ToBinary())
    if not mc.GetNumConformers():
        rdDepictor.Compute2DCoords(mc)
    return mc

def moltosvg(mol,molSize=(200,100),kekulize=True,drawer=None,**kwargs):
    mc = _prepareMol(mol,kekulize)
    if drawer is None:
        drawer = rdMolDraw2D.MolDraw2DSVG(molSize[0],molSize[1])
    drawer.DrawMolecule(mc,**kwargs)
    drawer.FinishDrawing()
    svg = drawer.GetDrawingText()
    return SVG(svg.replace('svg:',''))

def bokeh_outlier_plot(df,title='',fig_size=(600,600),legend_pos="bottom_right"):
    """
    df['standardized smiles'] -> list of smiles
    df['x'] -> x-coordinates
    df['y'] -> y-coordinates
    df['outl'] -> label of outlier detection
    df['colors'] -> color of outlier detection
    """
    lw = 1
    
    #smiles_df = pd.DataFrame(df['standardized smiles'].tolist(),columns =['SMILES'])
    PandasTools.AddMoleculeColumnToFrame(df,smilesCol='standardized smiles')
    #svgs = np.array([moltosvg(m).data for m in df.ROMol])
    #svgs = np.array([moltosvg(m) for m in df.ROMol])
    ChangeMoleculeRendering(renderer='PNG')

    hover = HoverTool(tooltips="""
        <div>
            <div>
                <span style="font-size: 8px;">SMILES: @desc</span>
            </div>
            <div>
                <span style="font-size: 10px; font-weight: bold;">Outlier?: @outl</span>
            </div>
            <div>
                <span style="font-size: 10px; font-weight: bold;">Dataframe index: @index</span>
            </div>
            <div>
            <span style="font-size: 10px;">X-coordinate: @y</span>
            </div>
            <div>
            <span style="font-size: 10px;">Y-coordinate: @x </span>
            </div>
        </div>
        """
    ,names=['outliers?'])
    
    fig = figure(plot_width=fig_size[0], plot_height=fig_size[1], tools=['reset,box_zoom,wheel_zoom,zoom_in,zoom_out,pan,save',hover])                         
    #fig.add_layout(Title(text=f'{metrics}', text_font_style="italic"), 'above')
    fig.add_layout(Title(text=f'{title}', text_font_style="italic"), 'above')
    fig.add_layout(Title(text=f'Outlier Visualization', text_font_size="16pt"), 'above')

    #source = ColumnDataSource(data=dict(x=df['x'], y=df['y'],c=df['colors'],outl=df['outl'], desc= df['standardized smiles'],svgs=svgs,index=df.index))
    source = ColumnDataSource(data=dict(x=df['x'], y=df['y'],c=df['colors'],outl=df['outl'], desc= df['standardized smiles'],index=df.index))

    r=fig.circle('x', 'y', size=7, source=source, name=f'outliers?', line_color='black', fill_color='c', fill_alpha=0.5, line_width=0.5,legend_field='outl')
    
    fig.background_fill_color = "lightgray"
    fig.background_fill_alpha = 0.3
    fig.legend.border_line_width = 2
    fig.legend.border_line_color = "black"
    fig.legend.background_fill_color = "lightgray"
    fig.legend.background_fill_alpha = 0.4                     
    fig.legend.label_text_font_size = "10px"
    
    fig.xaxis.axis_label ="X-coordinate"
    fig.yaxis.axis_label ="Y-coordinate"
    fig.legend.location = legend_pos

    return fig

### PCA dim red for plotting

In [ ]:
from sklearn.decomposition import PCA

pca_2 = PCA(n_components=2)
X_coor = pca_2.fit_transform(X)

#preparing df
df['x']=X_coor[:,0]
df['y']=X_coor[:,1]
df['outl']=labels
df['colors']=colors
bokeh_fig=bokeh_outlier_plot(df,title='Bottleneck features transformed using PCA(2)',fig_size=(600,600),legend_pos="bottom_right")
show(bokeh_fig)

### TSNE for plotting

In [ ]:
from sklearn.manifold import TSNE
pca_50 = PCA(n_components=50)
X_50 = pca_50.fit_transform(X)
X_embedded = TSNE(n_components=2, learning_rate='auto',init='random', perplexity=30).fit_transform(X_50)
df['x']=X_embedded[:,0]
df['y']=X_embedded[:,1]
bokeh_fig=bokeh_outlier_plot(df,title='Bottleneck features transformed using PCA(50) before TSNE',fig_size=(1200,1000),legend_pos="bottom_right")
show(bokeh_fig)

In [ ]:
#removing smiles with indices 13,58
indices=[13,58]
print(df['standardized smiles'][indices])
print(len(df))
df=df.drop(indices)
print(len(df))

In [ ]:
df.reset_index(inplace=True)
df.to_csv('clean_data.csv',index=False)